[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# NumPy y pandas esenciales, tipos de variables.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Lunes 19 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo I: *Datos y su calidad*

> 🔑 **Versión con soluciones** — solo para el instructor / revisión posterior.

## 🎯 Objetivo de la sesión
Cargar y explorar un `DataFrame` con soltura, y **clasificar cada variable por su tipo y por cómo se recolectó**.

Al terminar podrás:
1. Usar arreglos de *NumPy*: `dtype`, operaciones vectorizadas, *broadcasting* y máscaras booleanas.
2. Leer y explorar datos con *pandas*: `info`, `describe`, `loc`/`iloc`, filtros y `assign`.
3. Distinguir variables numéricas (continuas/discretas), categóricas (nominales/ordinales) y fechas, y representarlas con el `dtype` correcto (`category`, tipos *nullable*).
4. Reconocer sesgos de recolección: selección, supervivencia, no respuesta y medición.

## 🗺️ Agenda (120 min)
| Min | Bloque |
|---|---|
| 10 | Bienvenida, reglas de evaluación y asistencia, arranque del Codespace |
| 25 | 1. *NumPy* esencial |
| 35 | 2. *pandas* esencial |
| 25 | 3. Tipos de variables |
| 15 | 4. Sesgos de recolección |
| 10 | 5. Ejercicio guiado (diccionario de datos) y cierre |

## 🧭 Cómo usar este notebook
- Ejecuta las celdas **en orden** (`Shift + Enter`).
- 🟢 **Ejercicio básico**: lo que todos debemos dominar hoy.
- 🔴 **Reto (opcional)**: para quien termina antes o quiere profundizar. Si no te da tiempo en clase, inténtalo como práctica.
- Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 0. Bienvenida y reglas del curso

**Diploma:** se obtiene con **calificación final ≥ 70** y **asistencia de al menos 7 de 8 sesiones** (80%; se permite como máximo una falta).

| Evaluación | Se publica | Cierra | Composición |
|---|---|---|---|
| Evaluación semanal 1 (Módulos I y II) | jue 22 oct, 8:30 pm | dom 25 oct, 11:59 pm | Cuestionario 40 + notebook 60 |
| Evaluación semanal 2 (Módulos III y IV) | jue 29 oct, 8:30 pm | dom 1 nov, 11:59 pm | Cuestionario 30 + Proyecto integrador 70 |

- **Calificación final = (E1 + E2) / 2.**
- Entregas tardías: hasta 48 h con 10% de penalización; después ya no se califican.
- Los notebooks 01–06 son **práctica sin calificación**, pero son la mejor preparación para las evaluaciones.

**Arranque del Codespace:** en el repositorio del curso → botón verde **Code** → pestaña **Codespaces** → **Create codespace on main**. La primera vez tarda unos minutos porque instala `requirements.txt`. Después abre este notebook y elige el kernel **Python 3.12**.

### Preparación del entorno
Importamos las librerías y ubicamos la carpeta `data/`, que ya trae los CSV del curso (no hace falta internet).

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ubica la carpeta data/ tanto desde la raíz del repositorio como desde soluciones/
DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)  # semilla fija: resultados reproducibles

print("Datos en:", DATA)
print("NumPy", np.__version__, "| pandas", pd.__version__, "| seaborn", sns.__version__)

---
## 1. *NumPy* esencial (25 min)

*NumPy* es la base de casi todo el ecosistema científico de *Python* (*pandas*, *SciPy*, *statsmodels* y *Matplotlib* lo usan por dentro). Su objeto central es el **arreglo** (`ndarray`):
- **Homogéneo:** todos los elementos tienen el mismo `dtype` (`int64`, `float64`, `bool`, …).
- **De tamaño fijo y contiguo en memoria**, lo que permite operaciones **vectorizadas** escritas en C.

### 1.1 Arreglos y `dtype`

In [ ]:
# Masas corporales (g) de 8 pingüinos, como lista de Python y como arreglo de NumPy
masas_lista = [3750, 3800, 3250, 3450, 3650, 3625, 4675, 3475]
masas = np.array(masas_lista)

print(masas, type(masas))
print("dtype:", masas.dtype, "| forma:", masas.shape, "| dimensiones:", masas.ndim)

# Si mezclamos enteros y flotantes, NumPy "promueve" todo a float64
mixto = np.array([1, 2.5, 3])
print(mixto, mixto.dtype)

# np.nan es un flotante: un arreglo con faltantes no puede ser int64
con_faltante = np.array([3750, np.nan, 3250])
print(con_faltante, con_faltante.dtype)

### 1.2 Vectorización
Operar sobre el arreglo completo, sin bucles explícitos. Es más legible y mucho más rápido.

In [ ]:
# Convertir gramos a kilogramos: con bucle vs. vectorizado
kg_bucle = [m / 1000 for m in masas_lista]   # Python puro
kg_vector = masas / 1000                     # NumPy: una sola operación sobre todo el arreglo
print(kg_vector)

# Funciones de agregación
print("media:", masas.mean(), "| desv. est.:", masas.std(ddof=1).round(1), "| máx:", masas.max())

In [ ]:
# ¿Cuánto más rápido? Comparemos con un millón de valores simulados
grande = rng.normal(4200, 800, size=1_000_000)
grande_lista = grande.tolist()

%timeit -n 3 -r 3 [x / 1000 for x in grande_lista]
%timeit -n 3 -r 3 grande / 1000

### 1.3 *Broadcasting*
Cuando operamos arreglos de formas distintas, *NumPy* "estira" virtualmente el más pequeño si las dimensiones son compatibles (iguales o una de ellas vale 1, comparando de derecha a izquierda).

In [ ]:
# Matriz 4x3: filas = pingüinos, columnas = (pico_largo, pico_profundidad, aleta) en mm
medidas = np.array([
    [39.1, 18.7, 181],
    [39.5, 17.4, 186],
    [46.5, 17.9, 192],
    [50.0, 15.2, 218],
])
print("forma:", medidas.shape)

medias_col = medidas.mean(axis=0)          # forma (3,): una media por columna
print("medias por columna:", medias_col)

centradas = medidas - medias_col           # (4,3) - (3,) -> se "repite" la fila de medias
print(centradas.round(2))

# Para operar por FILA necesitamos una columna (4,1): se usa np.newaxis o reshape
medias_fila = medidas.mean(axis=1)[:, np.newaxis]
print((medidas - medias_fila).shape)

### 1.4 Máscaras booleanas
Una comparación produce un arreglo de `True/False` que sirve para **filtrar** y para **contar** (`True` vale 1).

In [ ]:
pesados = masas > 3600            # máscara booleana
print(pesados)
print("valores > 3600:", masas[pesados])
print("¿cuántos?", pesados.sum(), "| proporción:", pesados.mean())

# Combinar condiciones: & (y), | (o), ~ (no). ¡Siempre con paréntesis!
print(masas[(masas > 3400) & (masas < 3800)])

# np.where: valor condicional elemento a elemento
print(np.where(masas > 3600, "pesado", "ligero"))

### 🟢 Ejercicio 1 · Máscaras y faltantes
Con el arreglo `masa_g` (la masa de **todos** los pingüinos del dataset, incluye `NaN`):
1. Conviértelo a kilogramos en `masa_kg`.
2. Calcula la media **ignorando faltantes** (pista: `np.nanmean`). ¿Qué pasa si usas `.mean()`?
3. Cuenta cuántos pingüinos pesan más de 4.5 kg y qué proporción representan **entre los que tienen dato**.

In [ ]:
# Celda proporcionada: extraemos la columna como arreglo de NumPy
masa_g = pd.read_csv(os.path.join(DATA, "penguins.csv"))["body_mass_g"].to_numpy()
print(masa_g[:6], masa_g.dtype, masa_g.shape)

In [ ]:
masa_kg = masa_g / 1000
print("media con .mean():", masa_kg.mean())        # nan: un solo NaN "contamina" el resultado
print("media con np.nanmean():", round(np.nanmean(masa_kg), 3), "kg")

con_dato = ~np.isnan(masa_kg)                       # máscara de valores observados
mas_de_45 = masa_kg[con_dato] > 4.5
print("pesan más de 4.5 kg:", mas_de_45.sum())
print("proporción entre los que tienen dato:", round(mas_de_45.mean(), 3))
# Ojo: (masa_kg > 4.5).mean() dividiría entre TODOS (incluidos los NaN) y subestimaría la proporción.

### 🟢 Ejercicio 2 · Estandarizar con *broadcasting*
Estandariza cada **columna** de `medidas` (puntaje z: restar la media y dividir entre la desviación estándar de su columna) **sin bucles**. Verifica que las medias resultantes sean ≈ 0 y las desviaciones ≈ 1.

In [ ]:
z = (medidas - medidas.mean(axis=0)) / medidas.std(axis=0, ddof=1)
print(z.round(3))
print("medias:", z.mean(axis=0).round(10))
print("desv. est.:", z.std(axis=0, ddof=1).round(10))

### 🔴 Reto 1 · Matriz de distancias sin bucles
Calcula la matriz 4×4 de **distancias euclidianas** entre todos los pares de pingüinos de `z` usando solo *broadcasting* (pista: `z[:, None, :] - z[None, :, :]` tiene forma `(4, 4, 3)`). ¿Qué par de pingüinos es el más parecido?

In [ ]:
dif = z[:, None, :] - z[None, :, :]          # (4,1,3) - (1,4,3) -> (4,4,3)
dist = np.sqrt((dif ** 2).sum(axis=-1))      # (4,4)
print(dist.round(2))

# El par más parecido: ignoramos la diagonal (distancia 0 consigo mismo)
d = dist.copy()
np.fill_diagonal(d, np.inf)
i, j = np.unravel_index(d.argmin(), d.shape)
print(f"Par más parecido: pingüinos {i} y {j} (distancia {dist[i, j]:.2f})")

---
## 2. *pandas* esencial (35 min)

*pandas* agrega a *NumPy* **etiquetas** (índice y nombres de columna) y **columnas de tipos distintos**:
- `Series`: una columna (arreglo 1D + índice).
- `DataFrame`: una tabla; cada columna es una `Series` con su propio `dtype`.

### 2.1 Series y DataFrame

In [ ]:
s = pd.Series([3750, 3800, np.nan], index=["a", "b", "c"], name="body_mass_g")
print(s, "\n")
print("s['b'] =", s["b"], "| media (ignora NaN por defecto):", s.mean())

mini = pd.DataFrame({
    "species": ["Adelie", "Gentoo", "Chinstrap"],
    "body_mass_g": [3750, 5000, 3700],
})
mini

### 2.2 Lectura y primer vistazo: el dataset `penguins`
**Palmer Penguins** (Gorman, Williams & Fraser, 2014): medidas de 344 pingüinos de tres especies en tres islas del archipiélago Palmer (Antártida), registradas en las temporadas 2007–2009 por el programa *Palmer Station LTER*. Será nuestro dataset principal de la sesión 1 a la 6.

| Columna | Descripción |
|---|---|
| `species` | Especie: Adelie, Chinstrap, Gentoo |
| `island` | Isla: Biscoe, Dream, Torgersen |
| `bill_length_mm` | Largo del pico (culmen), mm |
| `bill_depth_mm` | Profundidad del pico, mm |
| `flipper_length_mm` | Largo de la aleta, mm |
| `body_mass_g` | Masa corporal, g |
| `sex` | Sexo (en este CSV: `MALE`/`FEMALE`) |

In [ ]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
print("forma (filas, columnas):", pinguinos.shape)
pinguinos.head()

In [ ]:
# info(): tipos, conteo de no nulos y memoria. Es lo PRIMERO que hay que mirar.
pinguinos.info()

In [ ]:
# describe(): resumen numérico; con include="all" también resume las columnas de texto
pinguinos.describe(include="all").T

In [ ]:
# Conteos de una variable categórica
pinguinos["species"].value_counts(dropna=False)

### 2.3 Selección: `[]`, `loc` e `iloc`
- `df["col"]` → una columna (Series); `df[["c1", "c2"]]` → varias (DataFrame).
- `df.loc[filas, columnas]` → por **etiqueta** (o máscara booleana). El rango `a:b` **incluye** `b`.
- `df.iloc[filas, columnas]` → por **posición** entera. El rango `a:b` **excluye** `b`.

In [ ]:
print(pinguinos.loc[0:2, ["species", "body_mass_g"]], "\n")   # filas con etiqueta 0, 1 y 2
print(pinguinos.iloc[0:2, [0, 5]])                            # posiciones 0 y 1

### 2.4 Filtros

In [ ]:
# Máscara booleana, igual que en NumPy
gentoo_pesados = pinguinos[(pinguinos["species"] == "Gentoo") & (pinguinos["body_mass_g"] > 5500)]
print(len(gentoo_pesados), "Gentoo de más de 5.5 kg")

# isin para varias categorías; query como alternativa legible
print(pinguinos[pinguinos["island"].isin(["Dream", "Torgersen"])].shape)
pinguinos.query("species == 'Chinstrap' and flipper_length_mm >= 205")

### 2.5 Crear columnas con `assign` y encadenar métodos
`assign` devuelve un **nuevo** DataFrame (no modifica el original) y permite escribir el análisis como una "receta" legible de pasos encadenados.

In [ ]:
resumen = (
    pinguinos
    .assign(body_mass_kg=lambda d: d["body_mass_g"] / 1000)      # nueva columna
    .query("island == 'Biscoe'")                                   # filtro
    .loc[:, ["species", "sex", "body_mass_kg"]]                    # selección de columnas
    .sort_values("body_mass_kg", ascending=False)                  # orden
    .head()
)
resumen

### 🟢 Ejercicio 3 · Explorar con *pandas*
1. ¿Cuántos pingüinos hay de cada especie **en cada isla**? (pista: `value_counts` sobre dos columnas, o `pd.crosstab`). ¿Qué especie vive en las tres islas?
2. Con `loc`, obtén `species`, `island` y `body_mass_g` de los Gentoo con masa mayor a 5,000 g, ordenados de mayor a menor masa. ¿Cuántos son?

In [ ]:
# 1. Conteo por isla y especie
print(pd.crosstab(pinguinos["island"], pinguinos["species"]), "\n")
# Adelie es la única especie presente en las tres islas; Gentoo solo en Biscoe y Chinstrap solo en Dream.

# 2. Gentoo de más de 5000 g
gentoo_5k = (
    pinguinos
    .loc[(pinguinos["species"] == "Gentoo") & (pinguinos["body_mass_g"] > 5000),
         ["species", "island", "body_mass_g"]]
    .sort_values("body_mass_g", ascending=False)
)
print(len(gentoo_5k), "pingüinos")
gentoo_5k.head()

### 🟢 Ejercicio 4 · Nuevas variables con `assign`
Crea, en un solo `assign`, las columnas `bill_ratio = bill_length_mm / bill_depth_mm` y `body_mass_kg`. Muestra los 5 pingüinos con mayor `bill_ratio`. ¿De qué especie son? ¿Por qué crees que pasa?

In [ ]:
(
    pinguinos
    .assign(
        bill_ratio=lambda d: d["bill_length_mm"] / d["bill_depth_mm"],
        body_mass_kg=lambda d: d["body_mass_g"] / 1000,
    )
    .nlargest(5, "bill_ratio")
    .loc[:, ["species", "island", "bill_length_mm", "bill_depth_mm", "bill_ratio", "body_mass_kg"]]
)
# Todos son Gentoo: tienen el pico largo y delgado (poco profundo), así que el cociente es mayor.

**✅ Respuesta sugerida:**

Los 5 con mayor `bill_ratio` son **Gentoo**: su pico es largo y poco profundo (delgado), por lo que el cociente largo/profundidad es alto. La razón resume la **forma** del pico, no su tamaño.

### 🔴 Reto 2 · Asignación encadenada
El siguiente código intenta poner en `NaN` la masa de los pingüinos de Torgersen, pero **no modifica `copia`** y *pandas* lanza `SettingWithCopyWarning`. Explica por qué y corrígelo con `loc` en una sola instrucción.

```python
copia = pinguinos.copy()
copia[copia["island"] == "Torgersen"]["body_mass_g"] = np.nan
```

In [ ]:
copia = pinguinos.copy()
# El original hace DOS operaciones: copia[mascara] crea un DataFrame temporal (una copia)
# y la asignación ["body_mass_g"] = ... modifica ese temporal, que se descarta.
# Con loc, selección y asignación son UNA sola operación sobre el objeto original:
copia.loc[copia["island"] == "Torgersen", "body_mass_g"] = np.nan
print(copia.groupby("island")["body_mass_g"].apply(lambda s: s.isna().sum()))

---
## 3. Tipos de variables (25 min)

El **tipo estadístico** de una variable decide qué resúmenes, gráficas y pruebas tienen sentido. No es lo mismo que el `dtype` de *pandas*, pero conviene que coincidan.

| Tipo estadístico | Subtipo | Ejemplo | Resumen típico | `dtype` sugerido |
|---|---|---|---|---|
| **Numérica** | Continua | `body_mass_g`, `fare` | media, mediana, desv. est., IQR | `float64` |
| | Discreta (conteos) | `sibsp`, `size` (tips) | igual + frecuencias | `int64` / `Int64` (nullable) |
| **Categórica** | Nominal (sin orden) | `species`, `island`, `sex` | frecuencias, moda | `category` |
| | Ordinal (con orden) | `class` (First > Second > Third) | frecuencias, mediana, percentiles | `category` ordenada |
| **Fecha / tiempo** | | fecha de muestreo | rango, frecuencia por periodo | `datetime64[ns]` |
| **Identificador** | | `id_muestra` | ¡ninguno! (no se promedia) | `string` |

⚠️ Trampas comunes: códigos numéricos que en realidad son categorías (`pclass` = 1, 2, 3; códigos postales), conteos guardados como `float` solo porque hay faltantes, y `bool` que en realidad es una categoría binaria.

### 3.1 `category`: nominal y ordinal

In [ ]:
tipos = pinguinos.copy()

# Nominal: el orden de las categorías no significa nada
tipos["species"] = tipos["species"].astype("category")
print(tipos["species"].cat.categories.tolist())

# Ordinal: creamos una "talla" a partir de la masa (solo para ilustrar)
tipos["talla"] = pd.cut(
    tipos["body_mass_g"],
    bins=[0, 3500, 4500, np.inf],
    labels=["chico", "mediano", "grande"],
    ordered=True,                 # el orden SÍ importa
)
print(tipos["talla"].dtype)
print(tipos["talla"].value_counts(sort=False))

# Con categorías ordenadas se puede comparar y ordenar con sentido
print("talla >= mediano:", (tipos["talla"] >= "mediano").sum())

In [ ]:
# Ahorro de memoria: texto repetido (object) vs. category
antes = pinguinos["island"].memory_usage(deep=True)
despues = pinguinos["island"].astype("category").memory_usage(deep=True)
print(f"object: {antes:,} bytes | category: {despues:,} bytes")

### 3.2 Tipos *nullable*: enteros con faltantes
`flipper_length_mm` y `body_mass_g` son enteros en el origen, pero *pandas* los lee como `float64` porque tienen `NaN`. Los tipos *nullable* (`Int64`, `boolean`, `string`, con mayúscula) representan el faltante como `pd.NA` sin cambiar el tipo.

In [ ]:
print(pinguinos["flipper_length_mm"].dtype, "->", pinguinos["flipper_length_mm"].astype("Int64").dtype)
print(pinguinos["flipper_length_mm"].astype("Int64").head())

# convert_dtypes() propone automáticamente los tipos nullable
pinguinos.convert_dtypes().dtypes

### 3.3 Fechas
`penguins` no trae fechas, así que usamos un pequeño registro ficticio de salidas de campo.

In [ ]:
salidas = pd.DataFrame({
    "fecha": ["2007-11-11", "2008-11-09", "2009-11-18", "2009-12-01"],
    "isla": ["Torgersen", "Biscoe", "Dream", "Biscoe"],
})
print(salidas.dtypes, "\n")
salidas["fecha"] = pd.to_datetime(salidas["fecha"])     # texto -> datetime64
salidas = salidas.assign(
    temporada=salidas["fecha"].dt.year,
    dia_semana=salidas["fecha"].dt.day_name(),
    dias_desde_inicio=(salidas["fecha"] - salidas["fecha"].min()).dt.days,
)
salidas

### 🟢 Ejercicio 5 · Clasifica las variables de `titanic`
Carga `titanic.csv` y clasifica estas columnas por tipo estadístico: `survived`, `pclass`, `sex`, `age`, `sibsp`, `fare`, `embarked`, `class`, `deck`, `alone`. Anota también qué `dtype` les asignó *pandas* y si lo cambiarías. Responde en la celda de abajo (puedes usar una tabla).

In [ ]:
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
titanic.dtypes

**✅ Respuesta sugerida:**

| Columna | Tipo estadístico | dtype leído | dtype sugerido |
|---|---|---|---|
| `survived` | Categórica binaria (nominal) | int64 | `category` o `boolean` |
| `pclass` | Categórica **ordinal** (codificada con números) | int64 | `category` ordenada |
| `sex` | Categórica nominal | object | `category` |
| `age` | Numérica continua | float64 | `float64` |
| `sibsp` | Numérica discreta (conteo) | int64 | `int64` |
| `fare` | Numérica continua | float64 | `float64` |
| `embarked` | Categórica nominal | object | `category` |
| `class` | Categórica ordinal (First > Second > Third) | object | `category` ordenada |
| `deck` | Categórica (ordinal según la posición en el barco; muchos faltantes) | object | `category` |
| `alone` | Categórica binaria | bool | `bool` / `boolean` |

`pclass` y `class` son **la misma información**: hay redundancia.

### 🔴 Reto 3 · Ordinal en la práctica
Convierte `titanic["class"]` en categoría **ordenada** de forma que `First` sea la **más alta**. Después calcula, usando una comparación (`>=`), la proporción de pasajeros que viajaban en `Second` o superior, y la tasa de supervivencia por clase en el orden correcto.

In [ ]:
orden = pd.CategoricalDtype(["Third", "Second", "First"], ordered=True)   # de menor a mayor
titanic["class"] = titanic["class"].astype(orden)

print("Second o superior:", round((titanic["class"] >= "Second").mean(), 3))
print(titanic.groupby("class", observed=True)["survived"].mean().round(3))

---
## 4. Sesgos de recolección (15 min)

Antes de calcular nada hay que preguntar **¿cómo llegaron estos datos aquí?** Ningún método estadístico corrige un sesgo de recolección si no lo reconocemos.

| Sesgo | Qué ocurre | Ejemplo |
|---|---|---|
| **Selección** | La muestra no representa a la población de interés | Medir pingüinos solo en las colonias más accesibles |
| **Supervivencia** | Solo observamos a quienes "pasaron un filtro" | Analizar solo empresas que siguen operando; aviones que regresaron |
| **No respuesta** | Quienes no responden difieren de quienes sí | Encuesta de ingreso que contestan menos los de ingresos altos |
| **Medición** | El instrumento o el procedimiento distorsiona el valor | Báscula descalibrada; preguntas que inducen la respuesta; redondeos |

Veamos una simulación de **selección**: ¿qué pasaría si solo hubiéramos muestreado la isla Biscoe?

In [ ]:
completo = pinguinos.dropna(subset=["body_mass_g"])
solo_biscoe = completo[completo["island"] == "Biscoe"]

comparacion = pd.DataFrame({
    "todas las islas": completo["species"].value_counts(normalize=True),
    "solo Biscoe": solo_biscoe["species"].value_counts(normalize=True),
}).fillna(0).round(3)
print(comparacion, "\n")
print(f"Masa media — todas: {completo['body_mass_g'].mean():.0f} g | solo Biscoe: {solo_biscoe['body_mass_g'].mean():.0f} g")

La masa "media de los pingüinos" cambia unos 500 g solo por **dónde** muestreamos: Biscoe concentra a los Gentoo, que son los más grandes. La muestra no está "mal", pero **las conclusiones solo valen para la población que representa**.

### 🟢 Ejercicio 6 · Identifica el sesgo
Para cada caso, indica el sesgo principal y una forma de mitigarlo:
1. Una app de fitness concluye que sus usuarios bajan de peso analizando solo a quienes siguen activos después de 6 meses.
2. Una encuesta telefónica en horario laboral sobre hábitos de consumo.
3. Un estudio de propinas (`tips`) registrado por **un solo mesero** en **un solo restaurante**.
4. Un sensor de temperatura que redondea hacia abajo al entero más cercano.

**✅ Respuesta sugerida:**

1. **Supervivencia** (o abandono): quienes no bajaron de peso tienden a dejar la app. Mitigar: analizar a todos los que empezaron (intención de tratar) y registrar los abandonos.
2. **Selección / no respuesta**: subrepresenta a quien trabaja en ese horario. Mitigar: varios horarios y canales, ponderar por perfil.
3. **Selección**: los resultados describen a ese mesero y ese restaurante, no a "los clientes" en general. Mitigar: muestrear varios restaurantes y meseros, o limitar las conclusiones.
4. **Medición** (sistemático): subestima en promedio ~0.5 °C. Mitigar: calibrar o corregir el sesgo conocido.

---
## 5. Ejercicio guiado: diccionario de datos de `penguins` (10 min + práctica)

Un **diccionario de datos** documenta cada columna: qué es, de qué tipo es, en qué unidades está y qué valores puede tomar. Es el primer entregable de cualquier EDA serio (y lo usarás en el proyecto integrador).

### 🟢 Ejercicio 7a · Construye el diccionario
Crea un DataFrame `diccionario` con una fila por columna de `pinguinos` y las columnas: `columna`, `tipo_estadistico`, `dtype_pandas`, `unidad`, `valores_posibles` (rango observado si es numérica; lista de categorías si es categórica) y `n_faltantes`.

*Pista:* las columnas `dtype_pandas`, `valores_posibles` y `n_faltantes` pueden calcularse con código; `tipo_estadistico` y `unidad` son conocimiento del dominio y se escriben a mano.

In [ ]:
tipo_estadistico = {
    "species": "categórica nominal",
    "island": "categórica nominal",
    "bill_length_mm": "numérica continua",
    "bill_depth_mm": "numérica continua",
    "flipper_length_mm": "numérica continua (registrada como entero)",
    "body_mass_g": "numérica continua (registrada como entero)",
    "sex": "categórica nominal (binaria)",
}
unidad = {"bill_length_mm": "mm", "bill_depth_mm": "mm", "flipper_length_mm": "mm", "body_mass_g": "g"}

def valores_posibles(s):
    # Rango para numéricas; lista de categorías para el resto
    if pd.api.types.is_numeric_dtype(s):
        return f"{s.min():g} – {s.max():g}"
    return ", ".join(sorted(s.dropna().unique()))

diccionario = pd.DataFrame({
    "columna": pinguinos.columns,
    "tipo_estadistico": [tipo_estadistico[c] for c in pinguinos.columns],
    "dtype_pandas": [str(pinguinos[c].dtype) for c in pinguinos.columns],
    "unidad": [unidad.get(c, "—") for c in pinguinos.columns],
    "valores_posibles": [valores_posibles(pinguinos[c]) for c in pinguinos.columns],
    "n_faltantes": pinguinos.isna().sum().to_numpy(),
})
diccionario

### 🟢 Ejercicio 7b · Convierte los `dtypes`
Crea `pinguinos_tipados` donde `species`, `island` y `sex` sean `category`, y `flipper_length_mm` y `body_mass_g` sean `Int64` (nullable). Verifica con `info()` y compara la memoria usada contra el original.

In [ ]:
pinguinos_tipados = pinguinos.astype({
    "species": "category",
    "island": "category",
    "sex": "category",
    "flipper_length_mm": "Int64",
    "body_mass_g": "Int64",
})
pinguinos_tipados.info()

mem_antes = pinguinos.memory_usage(deep=True).sum()
mem_despues = pinguinos_tipados.memory_usage(deep=True).sum()
print(f"\nMemoria: {mem_antes:,} -> {mem_despues:,} bytes ({mem_despues / mem_antes:.0%} del original)")

### 🟢 Ejercicio 7c · Pregunta de sesgo
Los datos se tomaron en **solo tres islas** del archipiélago Palmer y en **tres temporadas de anidación** (2007–2009). *Nota: la versión del CSV que usamos no incluye la columna del año; el dato viene de la documentación del dataset.*

¿Qué sesgo podría tener esta muestra? ¿Para qué población son válidas las conclusiones y qué **no** deberíamos afirmar?

**✅ Respuesta sugerida:**

- **Sesgo de selección / cobertura espacial:** tres islas no representan a todas las colonias de cada especie. Además las especies no están repartidas igual (Gentoo solo en Biscoe, Chinstrap solo en Dream), así que **especie e isla están confundidas**: una diferencia "entre islas" puede ser en realidad una diferencia entre especies, y viceversa.
- **Cobertura temporal:** tres temporadas consecutivas pueden coincidir con condiciones particulares (alimento, clima, hielo marino). No capturan tendencias de largo plazo.
- **Momento del muestreo / medición:** se mide durante la anidación; la masa corporal cambia a lo largo del ciclo reproductivo, así que la masa observada no es la de todo el año.
- **Supervivencia:** solo se miden adultos que llegaron a reproducirse.

Las conclusiones valen para **pingüinos reproductores de esas colonias en 2007–2009**. No deberíamos generalizar a "todos los Adelie de la Antártida" ni atribuir diferencias a la isla sin considerar la especie.

### 🔴 Reto 4 · Muestreo por conveniencia simulado
Simula que un equipo solo pudo medir a los **100 pingüinos más fáciles de atrapar**, y supón que la probabilidad de captura es mayor para los más ligeros (p. ej., proporcional a `1 / body_mass_g**3`). Con `rng.choice(..., p=..., replace=False)` toma esa muestra y compara la masa media y la proporción de especies con la del dataset completo. ¿De qué signo es el sesgo?

In [ ]:
base = pinguinos.dropna(subset=["body_mass_g"]).reset_index(drop=True)
peso = 1 / base["body_mass_g"].to_numpy() ** 3
p = peso / peso.sum()                           # probabilidades que suman 1

idx = rng.choice(len(base), size=100, replace=False, p=p)
muestra = base.iloc[idx]

print(f"Masa media — completo: {base['body_mass_g'].mean():.0f} g | muestra por conveniencia: {muestra['body_mass_g'].mean():.0f} g")
print(pd.DataFrame({
    "completo": base["species"].value_counts(normalize=True),
    "conveniencia": muestra["species"].value_counts(normalize=True),
}).round(3))
# El sesgo es NEGATIVO (subestima la masa) y además cambia la composición de especies (menos Gentoo).

---
## ✅ Cierre

**Hoy vimos:**
- *NumPy*: arreglos homogéneos, vectorización, *broadcasting* y máscaras booleanas.
- *pandas*: `read_csv`, `info`, `describe`, `loc`/`iloc`, filtros, `query` y `assign` encadenado.
- Tipos de variables y su `dtype`: `category` (nominal/ordinal), tipos *nullable* (`Int64`) y fechas.
- Sesgos de recolección: selección, supervivencia, no respuesta y medición.

**Práctica (sin calificación, en Classroom):** termina los ejercicios 🟢 que falten y, si puedes, un reto 🔴.

**Próxima sesión (mar 20 oct):** calidad de datos — duplicados, inconsistencias y valores faltantes (`02_calidad_y_faltantes.ipynb`). Ya viste que `penguins` tiene faltantes; mañana decidiremos qué hacer con ellos.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>